# Lab Assignment 9: Data Management Using `pandas`, Part 2
## DS 6001

### Instructions
Please answer the following questions as completely as possible using text, code, and the results of code as needed. Format your answers in a Jupyter notebook. To receive full credit, make sure you address every part of the problem, and make sure your document is formatted in a clean and professional way.

One of the biggest differences between using SQL and using pandas is the context around the data provenance. If you are using SQL, you are probably working with a relational database with a carefully constructed schema, in third (or a higher level) normal form, curated and organized to prevent time consuming and confusing data quality problems. If you are using pandas to work with raw data files, there are absolutely no guarantees regarding the quality of the data. You might be dealing with unnecessary columns, irrelevant rows, summary statistics included as additional observations, inscrutable column names, columns that should be rows and rows that should be columns, differences in the coverage of cases across data files, or differences in the ways the same observation is identified. You will have no choice but to contend with these issues as the clean, analytic data you generate at the end of all this wrangling may be exactly what you may need.

In this lab, we are going to contend with all of these problems using pandas. We are going to build the Country Analysis Relational DataBase (which we will call the C.A.R.D.B. or the "Cardi B"):

![CardbiB](https://media.giphy.com/media/3oEjI5ry4IwZ3RDw6k/giphy.gif "cardib")

We will be collecting data from two sources. First, we will use open data from the World Bank's [Sovereign
Environmental, Social, and Governance (ESG) Data](https://datatopics.worldbank.org/esg/) project. The ESG data reports data from every country in the world yearly since 1960 on a wide variety of topics including education, health, and economic factors within the countries. Second, we will use data on the quality and democratic character of countries' governments as reported by the [Varieties of Democracy (V-Dem)](https://www.v-dem.net/data/the-v-dem-dataset/) project at the University of Notre Dame. By using both data sources, we can conduct analyses to see whether democratic openness leads to better societal outcomes for countries. We can also write queries to capture a wide range of information on countries' political parties, tax systems, and banking industries, for example. Or as Cardi B would say, "You in the club just to party, I'm there, I get paid a fee. I be in and out them banks so much, I know they're tired of me."

## Problem 0
Import the following packages:

In [91]:
import numpy as np
import pandas as pd
import requests
import os
import io
import zipfile

Both the World Bank and V-Dem store their data in zipped directories containing CSV files. Download the World Bank data into your current working directory by running the following code:

In [92]:
# url = 'https://databank.worldbank.org/data/download/ESG_CSV.zip'
# r = requests.get(url)
# z = zipfile.ZipFile(io.BytesIO(r.content))
# z.extractall()

And download the V-Dem data by running:

In [93]:
# url = 'https://v-dem.net/media/datasets/V-Dem-CY-FullOthers-v15_csv.zip'
# r = requests.get(url)
# z = zipfile.ZipFile(io.BytesIO(r.content))
# z.extractall()

After you've run this code successfully once, the files you need will be in your working directory and you should save time by switching these cells from "code" to "raw", or commenting out the code, so that they don't run again if you restart the kernel.

You will only need two of the files you've downloaded. Load the 'V-Dem-CY-Full+Others-v15.csv' file as `vdem` and the 'ESGCSV.csv' file as `wb`. 

In [94]:
vdem = pd.read_csv('V-Dem-CY-Full+Others-v15.csv', low_memory=False)
wb = pd.read_csv('ESGCSV.csv')

## Problem 1
First, let's focus on the `vdem` data. Use `pandas` methods to perform the following tasks:

### Part a
Keep only the 'country_text_id', 'country_name', 'year', 'v2x_polyarchy', and 'v2peedueq' columns. [4 points]

In [95]:
vdem_clean = vdem[['country_text_id', 'country_name', 'year', 'v2x_polyarchy', 'v2peedueq']]
vdem_clean

,country_text_id,country_name,year,v2x_polyarchy,v2peedueq
0,MEX,Mexico,1789,0.028,NaN
1,MEX,Mexico,1790,0.028,NaN
2,MEX,Mexico,1791,0.028,NaN
3,MEX,Mexico,1792,0.028,NaN
4,MEX,Mexico,1793,0.028,NaN
...,...,...,...,...,...
27908,SPD,Piedmont-Sardinia,1857,0.205,NaN
27909,SPD,Piedmont-Sardinia,1858,0.206,NaN
27910,SPD,Piedmont-Sardinia,1859,0.206,NaN
27911,SPD,Piedmont-Sardinia,1860,0.210,NaN


### Part b
Use the `.query()` method to keep only the rows in which year is greater than or equal to 1960 and less than or equal to 2023 (the World Bank data only covers years between 1960 and 2023, filtering rows here will help us with joining these two dataframes later). [4 points]

In [96]:
vdem_clean = vdem_clean.query('year >= 1960 and year <= 2023')

### Part c
Rename 'country_text_id' to 'country_code', 'country_name' to 'country_name_vdem', 'v2x_polyarchy' to 'democracy', and 'v2peedueq' to 'educational_equality'. [4 points]

In [97]:
vdem_clean = vdem_clean.rename({
    'country_text_id': 'country_code',
    'country_name': 'country_name_vdem',
    'v2x_polyarchy': 'democracy',
    'v2peedueq': 'educational_equality'
}, axis=1)
vdem_clean

,country_code,country_name_vdem,year,democracy,educational_equality
171,MEX,Mexico,1960,0.231,-1.445
172,MEX,Mexico,1961,0.232,-1.445
173,MEX,Mexico,1962,0.232,-1.445
174,MEX,Mexico,1963,0.232,-1.445
175,MEX,Mexico,1964,0.231,-1.445
...,...,...,...,...,...
26508,ZZB,Zanzibar,2019,0.266,1.479
26509,ZZB,Zanzibar,2020,0.271,1.430
26510,ZZB,Zanzibar,2021,0.285,1.756
26511,ZZB,Zanzibar,2022,0.294,2.427


### Part d
Sort the rows by 'country_code' and 'year' in ascending order. [4 points]

In [98]:
vdem_clean = vdem_clean.sort_values(['country_code', 'year'])
vdem_clean

,country_code,country_name_vdem,year,democracy,educational_equality
5491,AFG,Afghanistan,1960,0.082,-1.203
5492,AFG,Afghanistan,1961,0.083,-1.203
5493,AFG,Afghanistan,1962,0.083,-1.203
5494,AFG,Afghanistan,1963,0.086,-1.203
5495,AFG,Afghanistan,1964,0.139,-1.046
...,...,...,...,...,...
26508,ZZB,Zanzibar,2019,0.266,1.479
26509,ZZB,Zanzibar,2020,0.271,1.430
26510,ZZB,Zanzibar,2021,0.285,1.756
26511,ZZB,Zanzibar,2022,0.294,2.427


## Problem 2
Next focus on the World Bank `wb` dataset 'ESGData.csv'. Use `pandas` methods to perform the following tasks:

### Part a
Keep only the columns named 'Country Code', 'Country Name', and 'Indicator Code', or begin with '19' or '20'. (Don't type in all the years individually. Instead, use code that finds all columns that begin '19' or '20'.) [4 points]

In [99]:
cols_19 = list(wb.columns[wb.columns.str.startswith(('19', '20'))])
wb_clean = wb[['Country Code', 'Country Name', 'Indicator Code'] + cols_19]
wb_clean

,Country Code,Country Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,ARB,Arab World,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,86.082150,86.321160,86.491156,86.583767,86.523897,86.527129,86.503099,86.347769,86.449919,NaN
1,ARB,Arab World,EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,88.406777,88.661643,89.186171,90.351067,88.689841,89.928996,90.241777,90.434095,90.654518,91.645430
2,ARB,Arab World,NY.ADJ.DRES.GN.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.200009,6.245928,5.406433,6.465929,8.238448,7.278235,4.628360,NaN,NaN,NaN
3,ARB,Arab World,NY.ADJ.DFOR.GN.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.078484,0.089821,0.087373,0.099548,0.054242,0.059945,0.071353,NaN,NaN,NaN
4,ARB,Arab World,AG.LND.AGRI.ZS,NaN,27.845967,27.83858,27.859229,27.863412,27.88415,27.88504,...,39.838000,39.870863,39.936386,39.985215,39.970860,39.907787,39.968729,39.967337,40.875467,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16964,ZWE,Zimbabwe,ER.PTD.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26.700000,26.700000,27.200000,27.200000,27.200000,27.200000,27.200000,27.200000,27.200000,28.300000
16965,ZWE,Zimbabwe,AG.LND.FRLS.HA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16966,ZWE,Zimbabwe,SL.UEM.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.774000,5.377000,5.886000,6.344000,6.793000,7.373000,8.621000,9.540000,10.087000,8.759000
16967,ZWE,Zimbabwe,SP.UWT.TFRT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.382129,10.400000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Part b
Rename 'Country Code' to'country_code', 'Country Name' to 'country_name_wb', and 'Indicator Code' to 'feature'. [4 points]

In [100]:
wb_clean = wb_clean.rename({
    'Country Code': 'country_code',
    'Country Name': 'country_name_wb',
    'Indicator Code': 'feature'
}, axis=1)

### Part c
Use the `.query()` method to remove the rows in which 'country_name_wb' is equal to one of the entries in the folowing `noncountries` list: [4 points]

In [101]:
noncountries = ["Arab World", "Central Europe and the Baltics",
                "Caribbean small states",
                "East Asia & Pacific (excluding high income)",
                "Early-demographic dividend","East Asia & Pacific",
                "Europe & Central Asia (excluding high income)",
                "Europe & Central Asia", "Euro area",
                "European Union","Fragile and conflict affected situations",
                "High income",
                "Heavily indebted poor countries (HIPC)","IBRD only",
                "IDA & IBRD total",
                "IDA total","IDA blend","IDA only",
                "Latin America & Caribbean (excluding high income)",
                "Latin America & Caribbean",
                "Least developed countries: UN classification",
                "Low income","Lower middle income","Low & middle income",
                "Late-demographic dividend","Middle East, North Africa, Afghanistan & Pakistan",
                "Middle income",
                "Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",
                "North America","OECD members",
                "Other small states","Pre-demographic dividend",
                "Pacific island small states",
                "Post-demographic dividend",
                "Sub-Saharan Africa (excluding high income)",
                "Sub-Saharan Africa",
                "Small states","East Asia & Pacific (IDA & IBRD)",
                "Europe & Central Asia (IDA & IBRD)",
                "Latin America & Caribbean (IDA & IBRD)",
                "Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)","South Asia",
                "South Asia (IDA & IBRD)",
                "Sub-Saharan Africa (IDA & IBRD)",
                "Upper middle income", "World"]

In [102]:
wb_clean = wb_clean.query(f'country_name_wb not in @noncountries')

### Part d
The features in this dataset are given strange and incomprehensible codes such as 'EG.CFT.ACCS.ZS'. Use the `replace_map` dictionary, defined below, to recode all of these values with more descriptive names for each feature. [4 points]

In [103]:
replace_map = {
  "AG.LND.AGRI.ZS": "agricultural_land",
  "AG.LND.FRST.ZS": "forest_area",
  "AG.PRD.FOOD.XD": "food_production_index",
  "CC.EST": "control_of_corruption",
  "EG.CFT.ACCS.ZS": "access_to_clean_fuels_and_technologies_for_cooking",
  "EG.EGY.PRIM.PP.KD": "energy_intensity_level_of_primary_energy",
  "EG.ELC.ACCS.ZS": "access_to_electricity",
  "EG.ELC.COAL.ZS": "electricity_production_from_coal_sources",
  "EG.ELC.RNEW.ZS": "renewable_electricity_output",
  "EG.FEC.RNEW.ZS": "renewable_energy_consumption",
  "EG.IMP.CONS.ZS": "energy_imports",
  "EG.USE.COMM.FO.ZS": "fossil_fuel_energy_consumption",
  "EG.USE.PCAP.KG.OE": "energy_use",
  "EN.ATM.CO2E.PC": "co2_emissions",
  "EN.ATM.METH.PC": "methane_emissions",
  "EN.ATM.NOXE.PC": "nitrous_oxide_emissions",
  "EN.ATM.PM25.MC.M3": "pm2_5_air_pollution",
  "EN.CLC.CDDY.XD": "cooling_degree_days",
  "EN.CLC.GHGR.MT.CE": "ghg_net_emissions",
  "EN.CLC.HEAT.XD": "heat_index_35",
  "EN.CLC.MDAT.ZS": "droughts",
  "EN.CLC.PRCP.XD": "maximum_5-day_rainfall",
  "EN.CLC.SPEI.XD": "mean_drought_index",
  "EN.MAM.THRD.NO": "mammal_species",
  "EN.POP.DNST": "population_density",
  "ER.H2O.FWTL.ZS": "annual_freshwater_withdrawals",
  "ER.PTD.TOTL.ZS": "terrestrial_and_marine_protected_areas",
  "GB.XPD.RSDV.GD.ZS": "research_and_development_expenditure",
  "GE.EST": "government_effectiveness",
  "IC.BUS.EASE.XQ": "ease_of_doing_business_rank",
  "IC.LGL.CRED.XQ": "strength_of_legal_rights_index",
  "IP.JRN.ARTC.SC": "scientific_and_technical_journal_articles",
  "IP.PAT.RESD": "patent_applications",
  "IT.NET.USER.ZS": "individuals_using_the_internet",
  "NV.AGR.TOTL.ZS": "agriculture",
  "NY.ADJ.DFOR.GN.ZS": "net_forest_depletion",
  "NY.ADJ.DRES.GN.ZS": "natural_resources_depletion",
  "NY.GDP.MKTP.KD.ZG": "gdp_growth",
  "PV.EST": "political_stability_and_absence_of_violence",
  "RL.EST": "rule_of_law",
  "RQ.EST": "regulatory_quality",
  "SE.ADT.LITR.ZS": "literacy_rate",
  "SE.ENR.PRSC.FM.ZS": "gross_school_enrollment",
  "SE.PRM.ENRR": "primary_school_enrollment",
  "SE.XPD.TOTL.GB.ZS": "government_expenditure_on_education",
  "SG.GEN.PARL.ZS": "proportion_of_seats_held_by_women_in_national_parliaments",
  "SH.DTH.COMM.ZS": "cause_of_death",
  "SH.DYN.MORT": "mortality_rate",
  "SH.H2O.SMDW.ZS": "people_using_safely_managed_drinking_water_services",
  "SH.MED.BEDS.ZS": "hospital_beds",
  "SH.STA.OWAD.ZS": "prevalence_of_overweight",
  "SH.STA.SMSS.ZS": "people_using_safely_managed_sanitation_services",
  "SI.DST.FRST.20": "income_share_held_by_lowest_20pct",
  "SI.POV.GINI": "gini_index",
  "SI.POV.NAHC": "poverty_headcount_ratio_at_national_poverty_lines",
  "SI.SPR.PCAP.ZG": "annualized_average_growth_rate_in_per_capita_real_survey_mean_consumption_or_income",
  "SL.TLF.0714.ZS": "children_in_employment",
  "SL.TLF.ACTI.ZS": "labor_force_participation_rate",
  "SL.TLF.CACT.FM.ZS": "ratio_of_female_to_male_labor_force_participation_rate",
  "SL.UEM.TOTL.ZS": "unemployment",
  "SM.POP.NETM": "net_migration",
  "SN.ITK.DEFC.ZS": "prevalence_of_undernourishment",
  "SP.DYN.LE00.IN": "life_expectancy_at_birth",
  "SP.DYN.TFRT.IN": "fertility_rate",
  "SP.POP.65UP.TO.ZS": "population_ages_65_and_above",
  "SP.UWT.TFRT": "unmet_need_for_contraception",
  "VA.EST": "voice_and_accountability",
  "EN.CLC.CSTP.ZS": "coastal_protection",
  "SD.ESR.PERF.XQ": "economic_and_social_rights_performance_score",
  "EN.CLC.HDDY.XD": "heating_degree_days",
  "EN.LND.LTMP.DC": "land_surface_temperature",
  "ER.H2O.FWST.ZS": "freshwater_withdrawal",
  "EN.H2O.BDYS.ZS": "water_quality",
  "AG.LND.FRLS.HA": "tree_cover_loss",
}


In [104]:
wb_clean = wb_clean.replace(replace_map)
wb_clean.head(5)

,country_code,country_name_wb,feature,1960,1961,1962,1963,1964,1965,1966,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
3266,AFG,Afghanistan,access_to_clean_fuels_and_technologies_for_coo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26.100000,27.600000,28.800000,30.300000,31.400000,32.600000,33.800000,34.900000,36.100000,NaN
3267,AFG,Afghanistan,access_to_electricity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,89.500000,71.500000,97.700000,97.700000,93.400000,97.700000,97.700000,97.700000,85.300000,85.3
3268,AFG,Afghanistan,natural_resources_depletion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.315571,0.290261,0.363282,0.350879,0.401053,0.370131,0.243668,0.335935,NaN,NaN
3269,AFG,Afghanistan,net_forest_depletion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.216609,0.232762,0.284781,0.229822,0.237615,0.269353,0.237958,0.317732,NaN,NaN
3270,AFG,Afghanistan,agricultural_land,NaN,57.878356,57.955016,58.031676,58.116002,58.123668,58.192662,...,58.123668,58.123668,58.123668,58.123668,58.276988,58.276988,58.741548,58.741548,58.741548,NaN


## Problem 3
The `wb` dataset is strangely organized. The features are stored in the rows, when typically we would want these features to be columns. Also, years are stored in columns, when typically we would want years to be represented by different rows. We can repair this structure by reshaping the data. 

### Part a
First, use `pd.melt()` to reshape the data to turn the columns that refer to years into rows. [8 points]

In [106]:
wb_clean = pd.melt(wb_clean, id_vars=['country_code', 'country_name_wb', 'feature'], var_name='year', value_name='feature_value')

### Part b
Then rename `variable` to `year`, and reshape the data again by using the `.pivot_table()` method to turn the rows that refer to features into columns. Make sure that the country code, country name, and year are included in columns in the dataframe, and not a row index. [8 points]

In [ ]:
# variable already named year in last problem
wb_clean.dtypes

country_code        object
country_name_wb     object
feature             object
year                object
feature_value      float64
dtype: object

In [110]:
wb_clean_pivot = wb_clean.pivot_table(index=['country_name_wb', 'country_code', 'year'], columns='feature', values='feature_value')
wb_clean = pd.DataFrame(wb_clean_pivot.to_records())
wb_clean

,country_name_wb,country_code,year,access_to_clean_fuels_and_technologies_for_cooking,access_to_electricity,agricultural_land,agriculture,annual_freshwater_withdrawals,annualized_average_growth_rate_in_per_capita_real_survey_mean_consumption_or_income,cause_of_death,...,research_and_development_expenditure,rule_of_law,scientific_and_technical_journal_articles,strength_of_legal_rights_index,terrestrial_and_marine_protected_areas,tree_cover_loss,unemployment,unmet_need_for_contraception,voice_and_accountability,water_quality
0,Afghanistan,AFG,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1961,NaN,NaN,57.878356,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1962,NaN,NaN,57.955016,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1963,NaN,NaN,58.031676,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1964,NaN,NaN,58.116002,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12347,Zimbabwe,ZWE,2019,30.3,46.7,39.518358,9.819260,34.553543,NaN,47.647301,...,NaN,-1.311041,426.38,6.0,27.2,NaN,7.373,NaN,-1.164831,NaN
12348,Zimbabwe,ZWE,2020,30.5,52.7,39.754073,8.772859,40.046325,NaN,NaN,...,NaN,-1.337362,456.49,NaN,27.2,NaN,8.621,NaN,-1.114599,83.3
12349,Zimbabwe,ZWE,2021,30.5,49.0,39.385906,8.849899,40.046325,NaN,NaN,...,NaN,-1.282122,495.68,NaN,27.2,NaN,9.540,NaN,-1.141045,NaN
12350,Zimbabwe,ZWE,2022,30.8,50.1,39.489284,7.170550,NaN,NaN,NaN,...,NaN,-1.237028,563.05,NaN,27.2,NaN,10.087,NaN,-1.109503,NaN


### Part c
After these reshapes, the year column in the `wb` data frame is stored as a string. Convert this column to an integer data type. [4 points]

In [113]:
wb_clean['year'] = wb_clean['year'].astype(int)

In [114]:
wb_clean.dtypes

country_name_wb                                        object
country_code                                           object
year                                                    int64
access_to_clean_fuels_and_technologies_for_cooking    float64
access_to_electricity                                 float64
                                                       ...   
tree_cover_loss                                       float64
unemployment                                          float64
unmet_need_for_contraception                          float64
voice_and_accountability                              float64
water_quality                                         float64
Length: 74, dtype: object

## Problem 4
Next we will merge the `wb` data frame with the `vdem` data frame, matching on the 'country_code' and 'year' columns. 

### Part a
First, write a sentence stating whether you expect this merge to be one-to-one, many-to-one, one-to-many, or many-to-many, and describe your rationale. [4 points]

<strong>
I expect this merge to be one-to-one. In the vdem dataset there is one row for each country/year pair because each year only has one democracy and one educational equality score. In the wb dataset, the same is true because the features similiarly describe information that is specific to a year/country pair and which will only have been observed once for that pair.
</strong>

### Part b
Next, merge the two datasets together in a way that checks whether your expectation is met, and also allows you to see the rows that failed to match. [4 points]

In [134]:
merged_data = pd.merge(wb_clean, vdem_clean, validate='one_to_one', on=['country_code', 'year'], how='outer', indicator='matched')

### Part c
After this merge, use the `.value_counts()` method to see the total number of observations that were found in both datasets, the number found only in the left dataset, and the number found only in the right dataset. (If you entered the `wb` data frame into the merge function first, then "left_only" refers to the rows found in the World Bank but not V-Dem, and "right_only" refers to the rows found in V-Dem but not the World Bank.) There should be more than 10,000 rows that matched, but more than 2000 that were found in `wb` but not `vdem`, and more than 400 that were found in `vdem` but not `wb`.

Then conduct two data aggregations to help us investigate why these observations did not match:

* First use `.query()` to keep only the observations that were present in `wb` but not `vdem`. (These are the 'left_only' observations if you typed the World Bank data into the merge function first.) Use `.groupby()` to aggregate the data by both 'country_code' and 'country_name_wb'. Include the minimum and maximum values of 'year' for each country in this aggregated dataframe.

* Then use `.query()` to keep only the observations that were present in `vdem` data but not `wb`. Use `.groupby()` to aggregate the data by both 'country_code' and 'country_name_vdem'. Include the minimum and maximum values of 'year' for each country in this aggregated dataframe. [8 points]

In [139]:
merged_data.value_counts('matched')

matched
both          10320
left_only      2032
right_only      409
Name: count, dtype: int64

<strong>
Left only here is the number of rows found in the wb data, not the vdem data. Right only is the rows found in the vdem data, not the wb data.
</strong

In [169]:
# wb only data
merged_data.query("matched=='left_only'").groupby(['country_code','country_name_wb']).agg(min_year=('year', 'min'), max_year=('year', 'max')).reset_index()

,country_code,country_name_wb,min_year,max_year
0,AND,Andorra,1960,2023
1,ARE,United Arab Emirates,1960,1970
2,ARM,Armenia,1960,1989
3,ATG,Antigua and Barbuda,1960,2023
4,AZE,Azerbaijan,1960,1989
5,BGD,Bangladesh,1960,1970
6,BHS,"Bahamas, The",1960,2023
7,BIH,Bosnia and Herzegovina,1960,1991
8,BLR,Belarus,1960,1989
9,BLZ,Belize,1960,2023


In [168]:
# vdem only data
merged_data.query("matched=='right_only'").groupby(['country_code','country_name_vdem']).agg(min_year=('year', 'min'), max_year=('year', 'max')).reset_index()

,country_code,country_name_vdem,min_year,max_year
0,DDR,German Democratic Republic,1960,1990
1,HKG,Hong Kong,1960,2023
2,PSE,Palestine/West Bank,1967,2023
3,PSG,Palestine/Gaza,1960,2023
4,SML,Somaliland,1991,2023
5,TWN,Taiwan,1960,2023
6,VDR,Republic of Vietnam,1960,1975
7,XKX,Kosovo,1999,2023
8,YMD,South Yemen,1960,1990
9,ZZB,Zanzibar,1960,2023


### Part d
Here's where a deep understanding of the data becomes very important. There are two reasons why an observation may fail to match in a merge. One reason is a difference in spelling. Suppose that South Korea (which is also known as the Republic of Korea) is coded as SKO in the World Bank data and ROK in V-Dem. In this case, we should recode one or the other of SKO and ROK so that they match, otherwise we will lose the data on South Korea. But the second reason why observations might fail to match is due to differences in coverage in the data collection strategy: it is possible that a country wasn't included in one data's coverage, or that certain years for that country were not included. For differences in coverage, there's no way to manipulate the data to match, so we are out of luck and we have to either delete these observations or proceed with missing data from one of the data sources.

Ukraine, and for some reason Seychelles, have no data from the World Bank data for 2023. Please ignore those issues for now. But for the other unmatched rows, take a close look at the two data aggregation tables you generated in part (c), and answer the following questions:


* Do you see any countries that are present in both the unmatched World Bank rows and the unmatched V-Dem rows, but with different spellings?

* Do some digging on Wikipedia and other sources on the Internet. What do you think is the primary reason why some countries are present in the V-Dem data but not the World Bank? (You don't need to describe the reasoning for every country. Just dig until you see a general pattern and describe it here.)

* Do some more digging on Wikipedia and other sources on the Internet. What do you think is the primary reason why some countries are present in the World Bank data but not V-Dem?  (You don't need to describe the reasoning for every country. Just dig until you see a general pattern and describe it here.)

[8 points]

<strong>

1. I don't see any countries that are not matching properly due to spelling or naming.  
2. The vdem data appears to include countries that are disputed territory and/or not internationally/UN recognized sovereign nations. Many of these territories/countries are or were self-governed and maintain(ed) a level of independence that makes the analysis of democracy in those regions extremely important to be included. I speculate that there may be political or logistical reasons why the World Bank does not gather separate data for these countries. 
3. The absences in the vdem data are more varied, but it looks like countries with very small populations are less likely to be included (for example: Monaco, San Marino). It also looks like coutries and years with civil unrest where/when data collection might be harder are not present in VDEM. World Bank may have more extensive resources to collect data in those countries.

</strong>

### Part e
Once you are convinced that all of the unmatched observations are due to differences in the coverage of the data collection strategies of the World Bank and V-Dem, repeat the merge, dropping all unmatched observations. This time there is no need to validate the type of merge, and no need to define a variable to indicate matching. [4 points]

In [193]:
merged_data = pd.merge(wb_clean, vdem_clean, on=['country_code', 'year'])
merged_data

,country_name_wb,country_code,year,access_to_clean_fuels_and_technologies_for_cooking,access_to_electricity,agricultural_land,agriculture,annual_freshwater_withdrawals,annualized_average_growth_rate_in_per_capita_real_survey_mean_consumption_or_income,cause_of_death,...,strength_of_legal_rights_index,terrestrial_and_marine_protected_areas,tree_cover_loss,unemployment,unmet_need_for_contraception,voice_and_accountability,water_quality,country_name_vdem,democracy,educational_equality
0,Afghanistan,AFG,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan,0.082,-1.203
1,Afghanistan,AFG,1961,NaN,NaN,57.878356,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan,0.083,-1.203
2,Afghanistan,AFG,1962,NaN,NaN,57.955016,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan,0.083,-1.203
3,Afghanistan,AFG,1963,NaN,NaN,58.031676,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan,0.086,-1.203
4,Afghanistan,AFG,1964,NaN,NaN,58.116002,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan,0.139,-1.046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10315,Zimbabwe,ZWE,2019,30.3,46.7,39.518358,9.819260,34.553543,NaN,47.647301,...,6.0,27.2,NaN,7.373,NaN,-1.164831,NaN,Zimbabwe,0.292,1.019
10316,Zimbabwe,ZWE,2020,30.5,52.7,39.754073,8.772859,40.046325,NaN,NaN,...,NaN,27.2,NaN,8.621,NaN,-1.114599,83.3,Zimbabwe,0.294,1.669
10317,Zimbabwe,ZWE,2021,30.5,49.0,39.385906,8.849899,40.046325,NaN,NaN,...,NaN,27.2,NaN,9.540,NaN,-1.141045,NaN,Zimbabwe,0.289,1.669
10318,Zimbabwe,ZWE,2022,30.8,50.1,39.489284,7.170550,NaN,NaN,NaN,...,NaN,27.2,NaN,10.087,NaN,-1.109503,NaN,Zimbabwe,0.286,1.617


## Problem 5
Write code using `pandas` that answers the next two questions:

### Part a
Of all countries in the data, since 1960, which countries have the highest and lowest average levels of democratic quality? [10 points]

In [185]:
merged_data.groupby(['country_code', 'country_name_wb']).agg(avg_democracy=('democracy', 'mean')).sort_values('avg_democracy')

,,avg_democracy
country_code,country_name_wb,
SAU,Saudi Arabia,0.015422
QAT,Qatar,0.026281
ARE,United Arab Emirates,0.043566
OMN,Oman,0.067328
ERI,Eritrea,0.081562
...,...,...
LUX,Luxembourg,0.870938
BEL,Belgium,0.872016
DEU,Germany,0.876188


<strong>
Between 1960 and 2023, Denmark has the highest average democratic quality and Saudi Arabia has the lowest average democratic quality. 
</strong>

### Part b
The 'educational_equality' index compiled by V-Dem measures the extent to which "high quality basic education guaranteed to all, sufficient to enable them to exercise their basic rights as adult citizens." They use a Bayesian scaling method to create a score for each country in each year that ranges roughly from -4 to 4, where low values of the scale mean that
> Provision of high quality basic education is extremely unequal and at least 75 percent (%) of children receive such low-quality education that undermines their ability to exercise their basic rights as adult citizens.

And high values mean that
> Basic education is equal in quality and less than five percent (%) of children receive such low-quality education that probably undermines their ability to exercise their basic rights as adult citizens.

Use the `pd.cut()` method to create a categorical version of 'educational_equality' with five categories, one from -4 to -2 called "extremely unequal", one from -2 to -.5 called "very unequal", one from -.5 to .5 called "somewhat unequal", one from .5 to 1.5 called "relatively equal", and one for values from 1.5 to 4 called "equal". (By default, the `pd.cut()` method sets `right=True`, which means the bins include their rightmost edges, so a value of exactly -2 will fall within the "extremely unequal" bin. Leave this default in place.)

Then aggregate the data to have one row per category of the new categorical version of "educational_equality". Collapse the following features to the mean with each category of "educational_equality":

* 'gini_index': The GINI index measures the amount of economic inequality in a country. The higher the index, the greater the economic disparity between rich and poor.
* 'poverty_headcount_ratio_at_national_poverty_lines': a measure of the proportion of the population living in poverty

Based on what you see, how would you describe the relationship between educational inequality, economic inequality, and poverty? (Don't worry about complete rigor with respect to causal inference, just describe what you see in this table.)

[10 points]
  

In [192]:
merged_data['educational_equality_cat'] = pd.cut(merged_data['educational_equality'], bins=[-4,-2,-.5, .5, 1.5, 4], labels=['extremely unequal', 'very unequal', 'somewhat unequal','relatively equal', 'equal'])
merged_data.groupby('educational_equality_cat').agg({'gini_index':'mean', 'poverty_headcount_ratio_at_national_poverty_lines': 'mean'})

/var/folders/fm/2_zxw4w57231pbsnw1dv90yc0000gn/T/ipykernel_20430/464610079.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged_data.groupby('educational_equality_cat').agg({'gini_index':'mean', 'poverty_headcount_ratio_at_national_poverty_lines': 'mean'})


,gini_index,poverty_headcount_ratio_at_national_poverty_lines
educational_equality_cat,,
extremely unequal,42.159091,59.322222
very unequal,45.627848,39.828017
somewhat unequal,43.250211,23.115126
relatively equal,36.496846,21.569167
equal,32.535973,17.419722


<strong>
Based on the above result, it appears that countries having a higher gini_index (i.e. more economic disparity) and a higher proportion of the population living in poverty correlates with having more unequal education. Shocking!
</strong>